# 01 — Data Audit & Identity Resolution  (Phase 1 EDA)

**Goal:** understand what the data actually supports *before* building anything.
This is exploratory data analysis only — **descriptive**, no thresholds, no models,
no detection-quality claims (the data is small dev/test data and unlabelled).

**Database:** `booktestdbgp_test` (MongoDB, read-only).

We work through seven questions, one section each:

1. **Inventory** — how much data, over what dates, how complete.
2. **Cross-game coverage** — do casino games produce bet records?
3. **Identity resolution** — does the phone-number join key work, and how often?
4. **Money** — deposits / withdrawals and where withdrawals are sent.
5. **IP semantics** — extracting the real client IP; device fingerprint.
6. **Bets sanity** — basic bet-outcome distribution.
7. **Coverage scorecard** — which fraud signals the data supports today.

> Reusable, unit-tested helpers (Mongo access, phone normalization, IP extraction)
> live in `src/frauddet/` and are imported below, so the notebook stays thin.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Find the repo root (folder containing 'src/') so we can import our package.
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import pandas as pd

from frauddet import config, io_mongo, identity, ip_utils

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("Repo root:", root)
print("Database :", config.get_database_name())

Repo root: C:\Users\dev\OneDrive\Documents\Fraud Detection - FS Book\Fraud-Detection-FS-Book
Database : booktestdbgp_test


In [2]:
# Connection smoke check (read-only). Expect {'ok': 1.0, ...}.
io_mongo.ping()

{'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1781170858, 28),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1781170858, 28)}

## 1. Inventory

Dev data is tiny, so we load each collection fully into a DataFrame (one connection)
and look at row counts, date ranges and field completeness.

In [3]:
with io_mongo.get_database() as db:
    players     = pd.DataFrame(list(db["players"].find()))
    bets        = pd.DataFrame(list(db["bet_transactions"].find()))
    deposits    = pd.DataFrame(list(db["deposittransactions"].find()))
    withdrawals = pd.DataFrame(list(db["withdrawaltransactions"].find()))
    gateway     = pd.DataFrame(list(db["gateway-payment-transactions"].find()))
    bonuses     = pd.DataFrame(list(db["bonustransactions"].find()))
    activity    = pd.DataFrame(list(db["useractivitylogs"].find()))
    logins      = pd.DataFrame(list(db["loginlogs"].find()))

frames = {
    "players": players,
    "bet_transactions": bets,
    "deposittransactions": deposits,
    "withdrawaltransactions": withdrawals,
    "gateway-payment-transactions": gateway,
    "bonustransactions": bonuses,
    "useractivitylogs": activity,
    "loginlogs": logins,
}
print({name: len(df) for name, df in frames.items()})

{'players': 61, 'bet_transactions': 342, 'deposittransactions': 130, 'withdrawaltransactions': 74, 'gateway-payment-transactions': 375, 'bonustransactions': 60, 'useractivitylogs': 3591, 'loginlogs': 787}


In [4]:
# Row counts and date range per collection.
date_fields = {
    "players": "createdAt",
    "bet_transactions": "createdDate",
    "deposittransactions": "createdAt",
    "withdrawaltransactions": "createdAt",
    "gateway-payment-transactions": "createdAt",
    "bonustransactions": "createdAt",
    "useractivitylogs": "created_at",
    "loginlogs": "createdAt",
}

summary = []
for name, df in frames.items():
    dates = pd.to_datetime(df[date_fields[name]], errors="coerce", utc=True)
    summary.append({"collection": name, "rows": len(df),
                    "first": dates.min(), "last": dates.max()})

pd.DataFrame(summary).set_index("collection")

,rows,first,last
collection,,,
players,61,2026-05-21 06:45:19.652000+00:00,2026-06-11 09:37:34.192000+00:00
bet_transactions,342,2026-05-26 05:32:50.833000+00:00,2026-06-11 09:20:53.561000+00:00
deposittransactions,130,2026-05-25 14:11:38.775000+00:00,2026-06-11 09:23:31.458000+00:00
withdrawaltransactions,74,2026-06-01 07:09:11.162000+00:00,2026-06-10 16:06:37.209000+00:00
gateway-payment-transactions,375,2026-05-11 13:21:38.841000+00:00,2026-06-11 09:23:31.438000+00:00
bonustransactions,60,2026-05-25 14:11:38.818000+00:00,2026-06-11 08:59:49.142000+00:00
useractivitylogs,3591,2026-05-21 06:39:18.872000+00:00,2026-06-11 09:39:42.247000+00:00
loginlogs,787,2026-05-21 06:43:12.403000+00:00,2026-06-11 09:39:17.249000+00:00


Field completeness (% of rows where the field is present). Showing only fields below 100% — these are the gaps worth knowing.

In [5]:
# players: which fields are sparse?
cov = (players.notna().mean() * 100).round(1)
cov[cov < 100].sort_values()

referredBy             3.3
referredCode           3.3
avatarImageName        3.3
avatarImageUrl         3.3
userClassification     6.6
address               18.0
passportDetails       36.1
kycUpdatedAt          36.1
emailId               45.9
isDeleted             45.9
nin                   60.7
countryCode           82.0
nationality           85.2
lastName              86.9
firstName             86.9
dtype: float64

In [6]:
# bets: which fields are sparse?
cov = (bets.notna().mean() * 100).round(1)
cov[cov < 100].sort_values()

freeBetId         0.0
resultedDate      2.0
result           95.9
fromAccountId    98.0
toAccountId      98.0
dtype: float64

## 2. Cross-game coverage  (the critical question)

The platform has a sportsbook **and** casino-style games. Do casino games write
bet records we can analyse, or only the sportsbook?

In [7]:
# What kinds of bets exist? Distinct values on key columns.
for col in ["gameType", "status", "currency", "betType", "result"]:
    print(f"{col:9}: {sorted(bets[col].dropna().unique().tolist())}")

gameType : ['Sports-book']
status   : ['ACCEPTED', 'SETTLED', 'VOID']
currency : ['INR']
betType  : ['multi', 'single']
result   : ['LOSE', 'VOID', 'WIN']


In [8]:
# Volume and total stake per game type.
bets.groupby("gameType").agg(n_bets=("_id", "size"), total_stake=("stake", "sum"))

,n_bets,total_stake
gameType,,
Sports-book,342,535699


In [9]:
# Casino side: is there a money trail, or only a catalog + telemetry?
with io_mongo.get_database() as db:
    casino = {
        "fundistgames (game catalog)": db["fundistgames"].count_documents({}),
        "game_shown":   db["game_shown"].count_documents({}),
        "game_clicked": db["game_clicked"].count_documents({}),
        "game_played":  db["game_played"].count_documents({}),
    }
pd.Series(casino, name="docs")

fundistgames (game catalog)    5980
game_shown                        2
game_clicked                     79
game_played                       2
Name: docs, dtype: int64

**Finding.** `bet_transactions` is **Sports-book only**. The casino side is a large
game *catalog* (`fundistgames`) plus a few *telemetry* events (`game_shown / clicked /
played`) that carry **no stake or payout** and key on a UUID `player_id` (not a phone).
So casino play has **no monetary records** and can't even be tied to a player today —
a logging gap to escalate.

## 3. Identity resolution & join rates

The de-facto join key across collections is the player's phone number, in three
formats (`0757…`, `757…`, `256757…`). `identity.normalize_phone` reduces all three
to the bare 9-digit subscriber number. First, is `username` really the same as
`contactNo`?

In [10]:
mismatches = (players["username"].astype(str) != players["contactNo"].astype(str)).sum()
print("username != contactNo mismatches:", int(mismatches), "of", len(players))

username != contactNo mismatches: 0 of 61


In [11]:
# Canonical set of player phones, then how well each feed joins to a player.
player_phones = set(players["username"].map(identity.normalize_phone).dropna())

join_fields = {
    "bet_transactions": ("loginId", bets),
    "deposittransactions": ("userId", deposits),
    "withdrawaltransactions": ("userId", withdrawals),
    "gateway-payment-transactions": ("userId", gateway),
    "bonustransactions": ("userId", bonuses),
    "useractivitylogs": ("playerId", activity),
}

rows = []
for name, (field, df) in join_fields.items():
    norm = df[field].map(identity.normalize_phone)
    matched = norm.isin(player_phones)
    rows.append({"collection": name, "field": field, "rows": len(df),
                 "matched": int(matched.sum()), "match_%": round(matched.mean() * 100, 1)})

pd.DataFrame(rows).set_index("collection")

,field,rows,matched,match_%
collection,,,,
bet_transactions,loginId,342,326,95.3
deposittransactions,userId,130,127,97.7
withdrawaltransactions,userId,74,74,100.0
gateway-payment-transactions,userId,375,209,55.7
bonustransactions,userId,60,53,88.3
useractivitylogs,playerId,3591,3312,92.2


In [12]:
# Who doesn't match? Sample of normalized IDs with no player (to decide keep-vs-drop later).
norm = activity["playerId"].map(identity.normalize_phone)
sorted(set(norm[~norm.isin(player_phones)].dropna()))[:10]

['6666699999',
 '702133848',
 '751010101',
 '751111111',
 '751231231',
 '751234569',
 '754222222',
 '754321012',
 '756969696',
 '758234902']

## 4. Money audit

In [13]:
print("Deposit status counts:")
print(deposits["status"].value_counts().to_string())
print("\nWithdrawal status counts:")
print(withdrawals["status"].value_counts().to_string())
print("\nWithdrawal executionType:")
print(withdrawals["executionType"].value_counts().to_string())

Deposit status counts:
status
completed                109
initiated                 11
manual_reconciliation      6
failed                     4

Withdrawal status counts:
status
initiated    35
completed    20
failed       11
declined      4
pending       4

Withdrawal executionType:
executionType
AUTO      50
MANUAL    24


In [14]:
print("Deposits by currency:")
print(deposits.groupby("currency")["amount"].agg(["count", "sum"]).to_string())
print("\nWithdrawals by currency:")
print(withdrawals.groupby("currency")["amount"].agg(["count", "sum"]).to_string())

Deposits by currency:
          count       sum
currency                 
UGX         130  15988850

Withdrawals by currency:
          count      sum
currency                
UGX          74  2935519


In [15]:
# Withdrawals sent to a different number than the player's own (third-party payout signal).
own   = withdrawals["userId"].map(identity.normalize_phone)
recip = withdrawals["recipientId"].map(identity.normalize_phone)
print("Withdrawals to a different number than own:", int((own != recip).sum()), "of", len(withdrawals))

# Recipient numbers that receive from more than one distinct player.
shared = withdrawals.assign(own=own, recip=recip).groupby("recip")["own"].nunique()
print("Recipient numbers shared by >1 player:", int((shared > 1).sum()))

Withdrawals to a different number than own: 5 of 74
Recipient numbers shared by >1 player: 0


In [16]:
# gateway-payment-transactions overlaps deposits by transactionId -> don't double-count deposits.
overlap = set(gateway["transactionId"]) & set(deposits["transactionId"])
print("Shared transactionIds (gateway & deposits):", len(overlap), "of", len(deposits), "deposits")

Shared transactionIds (gateway & deposits): 113 of 130 deposits


## 5. IP semantics & device fingerprint

`useractivitylogs.ip_address` is a **comma-separated string** (an X-Forwarded-For
chain), not an array. The real client is the first element that is **not** a
Cloudflare edge. `ip_utils` handles this in one tested place.

In [17]:
# Chain length distribution, and confirm position 0 = client, position 1 = Cloudflare.
chains = activity["ip_address"].map(ip_utils.split_chain)
print("Chain length distribution:")
print(chains.map(len).value_counts().sort_index().to_string())

first  = chains.map(lambda c: c[0] if len(c) >= 1 else None)
second = chains.map(lambda c: c[1] if len(c) >= 2 else None)
print("\n% of position 0 that is Cloudflare:", round(first.dropna().map(ip_utils.is_cloudflare).mean() * 100, 1))
print("% of position 1 that is Cloudflare:", round(second.dropna().map(ip_utils.is_cloudflare).mean() * 100, 1))

Chain length distribution:
ip_address
1     512
2    3079

% of position 0 that is Cloudflare: 0.0
% of position 1 that is Cloudflare: 100.0


In [18]:
# Extract the client IP, then count distinct players per client IP.
activity["client_ip"] = activity["ip_address"].map(ip_utils.extract_client_ip)
activity["phone"] = activity["playerId"].map(identity.normalize_phone)

print("Rows with a client IP:", int(activity["client_ip"].notna().sum()), "of", len(activity))
print("Distinct client IPs:", activity["client_ip"].nunique())
print("\nDistinct players per client IP (top 10) — note ::1/localhost & office IPs are dev artifacts:")
activity.dropna(subset=["client_ip"]).groupby("client_ip")["phone"].nunique().sort_values(ascending=False).head(10)

Rows with a client IP: 3591 of 3591
Distinct client IPs: 67

Distinct players per client IP (top 10) — note ::1/localhost & office IPs are dev artifacts:


client_ip
::1                28
80.227.165.206     25
5.30.200.3          6
58.84.62.122        5
58.84.62.68         5
58.84.60.94         5
58.84.62.145        4
110.227.210.234     3
91.75.84.76         2
58.84.61.244        2
Name: phone, dtype: int64

In [19]:
# Device fingerprint lives in loginlogs (not useractivitylogs).
print("loginlogs userType counts:")
print(logins["userType"].value_counts().to_string())
print("\nfingerprint coverage: %.1f%%" % (logins["fingerprint"].notna().mean() * 100))

players_in_logins = set(logins["loginId"].astype(str)) & set(players["username"].astype(str))
print("Players present in loginlogs:", len(players_in_logins), "of", len(players))

loginlogs userType counts:
userType
PLAYER        414
MANAGER       231
OPERATOR      127
SUPERVISOR     15

fingerprint coverage: 100.0%
Players present in loginlogs: 57 of 61


## 6. Bets sanity (descriptive only)

In [20]:
print("Result distribution:")
print(bets["result"].value_counts(dropna=False).to_string())
print("\nFree bets:", int(bets["isFreeBet"].sum()))
print("Bets with bonus stake (>0):", int((bets["stakeBonus"] > 0).sum()))
print("Currency mix:", bets["currency"].value_counts().to_dict())

settled = bets[bets["status"] == "SETTLED"]
win_rate = (settled["result"] == "WIN").mean() * 100
print(f"\nSettled bets: {len(settled)} | descriptive WIN rate: {win_rate:.1f}%  (NOT a performance metric)")

Result distribution:
result
LOSE    182
WIN     139
NaN      14
VOID      7

Free bets: 0
Bets with bonus stake (>0): 7
Currency mix: {'INR': 342}

Settled bets: 321 | descriptive WIN rate: 43.3%  (NOT a performance metric)


## 7. Coverage scorecard — what the data supports today

In [21]:
scorecard = [
    ("Sportsbook per-bet records", "YES", "bet_transactions, ticket level"),
    ("Casino per-round bet records", "NO", "telemetry only, no stake/payout"),
    ("Casino activity tied to a player", "NO", "game_* events use a UUID, not phone"),
    ("Client IP (Cloudflare stripped)", "YES", "useractivitylogs, first non-CF element"),
    ("Device fingerprint (players)", "YES", "loginlogs.fingerprint, ~100% coverage"),
    ("Deposits", "YES", "deposittransactions"),
    ("Withdrawals + recipient number", "YES", "withdrawaltransactions.recipientId"),
    ("Payment-instrument sharing detectable", "YES", "recipient is a phone number"),
    ("Bonus grants", "YES", "bonustransactions / bonusaccounts"),
    ("KYC status", "YES", "players.KycVerified"),
    ("Referral graph", "PARTIAL", "players.referredBy only ~3% filled"),
]
pd.DataFrame(scorecard, columns=["capability", "supported", "notes"]).set_index("capability")

,supported,notes
capability,,
Sportsbook per-bet records,YES,"bet_transactions, ticket level"
Casino per-round bet records,NO,"telemetry only, no stake/payout"
Casino activity tied to a player,NO,"game_* events use a UUID, not phone"
Client IP (Cloudflare stripped),YES,"useractivitylogs, first non-CF element"
Device fingerprint (players),YES,"loginlogs.fingerprint, ~100% coverage"
Deposits,YES,deposittransactions
Withdrawals + recipient number,YES,withdrawaltransactions.recipientId
Payment-instrument sharing detectable,YES,recipient is a phone number
Bonus grants,YES,bonustransactions / bonusaccounts


## Summary

- **Sportsbook only** has money records; casino has no per-round bet data and isn't
  tied to players → escalate the logging gap.
- The **phone join key works well** (withdrawals 100%, deposits ~98%, bets ~95%,
  activity ~92%); the gateway log is noisier (~53%) and overlaps deposits — use
  `deposittransactions` as the deposit ledger.
- **Client IP** is recoverable after stripping Cloudflare; **device fingerprint**
  exists in `loginlogs` for players.
- All numbers here are **descriptive** on tiny dev data — not performance metrics.

Next phase: build the flattening layer (clean parquet tables) on top of these
verified facts.